# ML Masterclass 10: Monitoring & Governance (MLOps)

Welcome to Notebook 09, the final module in the ML Masterclass.

Deployment is not the finish line; it is the starting line. Software engineers monitor CPU usage and latency. Machine Learning engineers must monitor **Data Drift**, **Model Decay**, and **Algorithmic Fairness**.

---

## 🎓 Socratic Deep Check
Ponder this question before we begin. The answer defines whether you retrain your model or rethink your entire business strategy:

> *"During COVID-19, human buying behavior changed completely. Is this Covariate Shift or Concept Drift? Do you need to update your model's weights, or just feed it new data?"*

---

## Table of Contents
1. **The Taxonomy of ML Outages:** Concept Drift vs. Covariate Shift vs. Prior Probability Shift.
2. **Population Stability Index (PSI):** Mathematically detecting drift without ground-truth labels.
3. **Interpretability & Governance (SHAP):** Explaining block-box decisions to regulators.


### 🎓 Junior to Senior: Concept Bridge

**The Senior Perspective:** Deployment is the starting line, not the finish line. In production, data distributions shift, user behavior changes, and model accuracy silently degrades. Seniors set up automated drift detection (PSI, KL-divergence) that triggers alerts BEFORE business metrics drop — because by the time you notice revenue falling, the model has been broken for weeks.

**Mental Model:** Think of model monitoring like a smoke detector. You don't wait for the house to burn down (business metrics crash) to discover the problem. PSI/drift detection is the smoke detector — it catches distribution shifts in the INPUT features immediately, before the fire (output degradation) spreads. No smoke detector = no warning = catastrophic silent failure.

**Common Junior Pitfall:** Only monitoring model accuracy. Accuracy requires ground truth labels, which often arrive days or weeks late (did the user actually churn? did the loan default?). By the time you compute accuracy, the damage is done. Monitor INPUT feature distributions (PSI) instead — drift is detectable in real-time without labels.

---


## 1. The Taxonomy of ML Outages

When a model's accuracy drops in production, it's rarely a code bug. It's usually a shift in the mathematical distribution of the universe.

1. **Covariate Shift (Data Drift):** $P(X)$ changes, but $P(y|X)$ stays the same. 
   * *Example:* You trained a face detector on high-quality studio photos ($X$). In production, users upload blurry webcam photos ($X'$). The definition of a human face hasn't changed, but the distribution of the inputs did.
   * *Fix:* Collect more webcam photos and fine-tune. The baseline logic is sound.
2. **Prior Probability Shift (Label Drift):** $P(y)$ changes, but $P(X|y)$ stays the same.
   * *Example:* A disease was super rare (1% rate). Suddenly, an epidemic hits, and the rate jumps to 20%. The symptoms ($X$) of the disease ($y$) are exactly the same, but the background probability of having it shifted.
   * *Fix:* Often fixed by recalibrating the final probability thresholds; no need to retrain the core feature extractors.
3. **Concept Drift:** $P(y|X)$ changes completely.
   * *Example:* In 2019, buying 100 rolls of toilet paper ($X$) reliably predicted you ran a restaurant/hotel ($y$). In March 2020, buying 100 rolls of toilet paper meant you were a panicked residential consumer.
   * *Fix:* **The model is dead.** The fundamental relationship between features and targets has changed. You must retrain from scratch on the new reality.

### 🎓 DEEP QUESTION ANSWERED
**Q:** *COVID-19 buying behavior: Covariate or Concept?*

**A:** It was massive **Concept Drift**. The mapping function $f(X) \rightarrow y$ broke entirely. Updating weights wouldn't fix it; the model's fundamental understanding of human economics was obsolete. Models had to be shut down and rewritten.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid")

# -----------------------------------------------------
# IMPLEMENTATION: Population Stability Index (PSI)
# -----------------------------------------------------
# PSI calculates how much a feature's distribution has shifted from Training to Production.
# Crucially, it does NOT require the 'Ground Truth' Y-label (which you often don't have in real-time).
# PSI = sum( (Actual_% - Expected_%) * ln(Actual_% / Expected_%) )

def calculate_psi(expected, actual, bins=10):
    """Calculates PSI for a single continuous feature."""
    # Bin the data based on the expected (training) distribution
    breakpoints = np.percentile(expected, np.linspace(0, 100, bins + 1))
    
    expected_percents = []
    actual_percents = []
    
    for i in range(bins):
        # Count how many fall into this bin
        if i == bins - 1: # Last bin includes maximum
            exp_count = np.sum((expected >= breakpoints[i]) & (expected <= breakpoints[i+1]))
            act_count = np.sum((actual >= breakpoints[i]) & (actual <= breakpoints[i+1]))
        else:
            exp_count = np.sum((expected >= breakpoints[i]) & (expected < breakpoints[i+1]))
            act_count = np.sum((actual >= breakpoints[i]) & (actual < breakpoints[i+1]))
            
        # Convert to percentages (add tiny epsilon to avoid div by zero / log zero)
        expected_percents.append(max(exp_count / len(expected), 0.0001))
        actual_percents.append(max(act_count / len(actual), 0.0001))
        
    expected_percents = np.array(expected_percents)
    actual_percents = np.array(actual_percents)
    
    psi_values = (actual_percents - expected_percents) * np.log(actual_percents / expected_percents)
    return np.sum(psi_values)

# 1. Stable Feature (No Drift)
train_feature_stable = np.random.normal(50, 10, 5000)
prod_feature_stable = np.random.normal(51, 10, 1000) # Tiny shift
psi_stable = calculate_psi(train_feature_stable, prod_feature_stable)

# 2. Drifted Feature (Covariate Shift)
train_feature_drift = np.random.normal(50, 10, 5000)
prod_feature_drift = np.random.normal(65, 15, 1000) # Massive shift in mean and variance
psi_drift = calculate_psi(train_feature_drift, prod_feature_drift)

print(f"PSI for Stable Feature: {psi_stable:.4f}  (< 0.1 is Excellent)")
print(f"PSI for Drifted Feature: {psi_drift:.4f}  (> 0.2 is Critical Red Alert!)\n")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
sns.kdeplot(train_feature_stable, ax=ax1, label='Training', fill=True)
sns.kdeplot(prod_feature_stable, ax=ax1, label='Production', fill=True)
ax1.set_title(f"Stable Feature (PSI: {psi_stable:.2f})")

sns.kdeplot(train_feature_drift, ax=ax2, label='Training', fill=True)
sns.kdeplot(prod_feature_drift, ax=ax2, label='Production', fill=True)
ax2.set_title(f"Drifted Feature (PSI: {psi_drift:.2f})")

plt.legend()
plt.show()

## 3. Interpretability & Governance (SHAP)

In highly regulated industries (Finance, Healthcare), deploying a black box Random Forest is illegal. If the model denies a loan, GDPR / ECOA laws dictate you must provide an exact reason *why*.

**SHAP (SHapley Additive exPlanations)** solves this using Cooperative Game Theory. 
It assumes the prediction is a payout, and the features are players in a game. It mathematically calculates exactly how much money (how much probability change) each feature contributed to the final prediction, ensuring the sum of all features equals the final output.

### 🎓 DEEP QUESTION ANSWERED
**Q:** *If an AI system denies a loan, how do SHAP values allow you to mathematically explain the decision of a black-box model to a regulator?*

**A:** 
SHAP calculates the "Base Value" (e.g., the average approval rate for everyone is 40%). 
Then, for John Doe's specific application, SHAP mathematically breaks down his unique path to his 5% approval rating:
* Base Value: 40%
* Feature: `Income=$40,000` -> Pushed probability down -15%
* Feature: `Credit_Score=600` -> Pushed probability down -25%
* Feature: `No_Prior_Defaults=True` -> Pushed probability up +5%
* Final Output: 40 - 15 - 25 + 5 = **5%**

You can hand this exact, mathematically rigorous breakdown to a regulator or the customer to prove exactly which features caused the denial.

---
# Machine Learning: Masterclass Complete 🎓

You have completed the exhaustive 0-to-Masterclass sequence for Machine Learning.
You have built AutoDiff engines from scratch, manipulated infinite-dimensional Kernel spaces, uncovered the data leakage that destroys production pipelines, and architected real-time streaming feature stores. 

You are no longer blindly calling `.fit()`. You are architecting intelligent systems.

---
## ✅ Knowledge Check

### Q1: Covariate Shift vs Concept Drift

<details><summary>👉 Answer</summary>

Covariate Shift: P(X) changes but P(Y|X) stays the same. Your face detector trained on studio photos sees blurry webcam photos — the definition of a face hasn't changed, just the input quality. Fix: add more diverse training data. Concept Drift: P(Y|X) changes entirely. During COVID, buying 100 toilet paper rolls changed meaning from 'hotel owner' to 'panicked consumer.' Fix: retrain from scratch on new reality — the old model is fundamentally wrong.
</details>

### Q2: PSI interpretation

<details><summary>👉 Answer</summary>

PSI < 0.1: no significant shift (stable). PSI 0.1-0.2: moderate shift (investigate). PSI > 0.2: critical shift (retrain immediately). PSI works WITHOUT ground truth labels — it compares the distribution of input features between training data and production data. This makes it ideal for real-time monitoring where labels arrive late.
</details>

### Q3: SHAP for regulatory compliance

<details><summary>👉 Answer</summary>

SHAP decomposes a prediction into per-feature contributions that sum to the final output. For loan denial: Base rate = 40% approval → Income contribution = -15% → Credit score = -25% → No prior defaults = +5% → Final = 5%. This satisfies GDPR/ECOA requirements that demand EXPLAINABLE AI decisions. Without SHAP, you can't deploy black-box models in regulated industries (finance, healthcare, insurance).
</details>


---
## 🔨 Practical Practice

### Exercise 1: Implement a complete drift detection pipeline: compute PSI for 5 features, set up alert thresholds (green/yellow/red), and simulate a gradual distribution shift over 12 months. Plot PSI trends over time and identify when the alert triggers.

### Exercise 2: Build a SHAP explainer for a trained XGBoost model on a loan approval dataset. Generate: (1) a global feature importance bar chart, (2) a beeswarm plot, and (3) a waterfall plot for a single denied application explaining exactly why it was denied.

### Exercise 3: Simulate concept drift: train a model on pre-2020 e-commerce data where 'bulk toilet paper purchase' predicts 'business customer.' Then evaluate on post-2020 data where the same feature predicts 'panicked consumer.' Show accuracy collapse and how retraining on new data fixes it.
